# IMPORTS

Text: The original token text.
Dep: The syntactic relation connecting child to head.
Head text: The original text of the token head.
Head POS: The part-of-speech tag of the token head.
Children: The immediate syntactic dependents of the token.

In [98]:
from transformers import BertTokenizer, BertModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import spacy
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"




In [84]:

nlp = spacy.load("en_core_web_sm")
doc = nlp("Autonomous cars shift insurance liability toward manufacturers")
for chunk in doc.noun_chunks:
    print(chunk.text, chunk.root.text, chunk.root.dep_,
            chunk.root.head.text)


Autonomous cars cars nsubj shift
insurance liability liability dobj shift
manufacturers manufacturers pobj toward


# Sample

### King and queen custom embedd

In [3]:
embedding_king = np.array([0.2, 0.5, 0.1])
embedding_queen = np.array([0.3, 0.4, 0.2])

embedding_king_reshaped = embedding_king.reshape(1, -1)
embedding_queen_reshaped = embedding_queen.reshape(1, -1)

# Calculate cosine similarity
cosine_sim = cosine_similarity(embedding_king_reshaped, embedding_queen_reshaped)

# Output the result
print(F"King : {embedding_king} \nQueen : {embedding_queen}")
print(f"Cosine Similarity between 'king' and 'queen': {cosine_sim[0][0]:.4f}")



King : [0.2 0.5 0.1] 
Queen : [0.3 0.4 0.2]
Cosine Similarity between 'king' and 'queen': 0.9493


### using LLM Distiled bert-l6v2

In [33]:
sen_tran = SentenceTransformer('paraphrase-MiniLM-L6-v2')
distBert = SentenceTransformer('all-MiniLM-L12-v2')  # Example small LLM for embeddings

/home/irshad/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [105]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
sen_tran = SentenceTransformer('paraphrase-MiniLM-L6-v2')
distBert = SentenceTransformer('distilbert-base-nli-stsb-mean-tokens')

def encode_with_bert(text):
    inputs = tokenizer(text, return_tensors='pt')
    outputs = bert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()

def sim(ans, key):
    data = []
    for model, name in zip([sen_tran, distBert, encode_with_bert], ["Sentence Transformer", "DistilBERT", "BERT"]):
        if name == "BERT":
            ans_emb = model(ans)
            key_emb = model(key)
        else:
            ans_emb = model.encode(ans)
            key_emb = model.encode(key)
        
        similarity = cosine_similarity([ans_emb], [key_emb])
        data.append({
            "Model": name,
            "Cosine Similarity Score": similarity[0][0],
            "Answer Key": key,
            "Student Answer": ans,
            "Embeddings Shape": np.shape(ans_emb)
        })
    
    df = pd.DataFrame(data)
    return df.sort_values(by="Cosine Similarity Score", ascending=False)

/home/irshad/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [104]:
answer = "I love natural language processing."
key = "Natural language processing is a fascinating field."
sim(answer, key)


,Model,Cosine Similarity Score,Answer Key,Student Answer,Embeddings Shape
0,Sentence Transformer,0.777803,Natural language processing is a fascinating f...,I love natural language processing.,"(384,)"
1,DistilBERT,0.812356,Natural language processing is a fascinating f...,I love natural language processing.,"(768,)"
2,BERT,0.772469,Natural language processing is a fascinating f...,I love natural language processing.,"(768,)"


In [ ]:
def sim(ans,key):
    for model,name in zip([sen_tran,distBert],["Sentence Transformer","DistilBERT"]):
        ans_emb = model.encode(ans)
        key_emb = model.encode(key)
        similarity = cosine_similarity([ans_emb], [key_emb])
        print(f"{name.upper()}")
        print("-"*len(name))
        print(f"\nAnswerkey: {key}\nStudent Answer: {ans}")
        # print(f"Embeddings: {key_emb}\nStudent Answer: {ans_emb}")
        print(f"Embeddings shape ={np.shape(ans_emb)}")
        print(f"Cosine Similarity Score: {similarity[0][0]}\n")

        

In [ ]:

student_answer = "The mitochondria is the powerhouse of the cell."
reference_answer = "generate energy for the cell."

# Generate embeddings
sim(student_answer,reference_answer)


,Model,Cosine Similarity Score,Answer Key,Student Answer,Embeddings Shape
2,BERT,0.722458,generate energy for the cell.,The mitochondria is the powerhouse of the cell.,"(768,)"
0,Sentence Transformer,0.518479,generate energy for the cell.,The mitochondria is the powerhouse of the cell.,"(384,)"
1,DistilBERT,0.509188,generate energy for the cell.,The mitochondria is the powerhouse of the cell.,"(768,)"


In [108]:
student_answer = "The mitochondria is the powerhouse of the cell."
reference_answer = "The mitochondria is not the powerhouse of the cell."
sim(student_answer,reference_answer)


,Model,Cosine Similarity Score,Answer Key,Student Answer,Embeddings Shape
2,BERT,0.965972,The mitochondria is not the powerhouse of the ...,The mitochondria is the powerhouse of the cell.,"(768,)"
0,Sentence Transformer,0.762137,The mitochondria is not the powerhouse of the ...,The mitochondria is the powerhouse of the cell.,"(384,)"
1,DistilBERT,0.459251,The mitochondria is not the powerhouse of the ...,The mitochondria is the powerhouse of the cell.,"(768,)"


In [109]:

# Load a pre-trained model
# Example text input
text1 = "man"
text2 = "men"
sim(text1,text2)


,Model,Cosine Similarity Score,Answer Key,Student Answer,Embeddings Shape
1,DistilBERT,0.947331,men,man,"(768,)"
2,BERT,0.659932,men,man,"(768,)"
0,Sentence Transformer,0.551262,men,man,"(384,)"


In [111]:
# Example text input
text1 = "I love machine learning"
text2 = "Machine learning is fascinating"
sim(text1,text2)


,Model,Cosine Similarity Score,Answer Key,Student Answer,Embeddings Shape
1,DistilBERT,0.904447,Machine learning is fascinating,I love machine learning,"(768,)"
2,BERT,0.834357,Machine learning is fascinating,I love machine learning,"(768,)"
0,Sentence Transformer,0.814237,Machine learning is fascinating,I love machine learning,"(384,)"


# Preprocessing


### Levenshtein

In [171]:
import Levenshtein

def calculate_levenshtein_distance(s1, s2):
    return Levenshtein.distance(s1, s2)

def evaluate_answer(student_answer, correct_answer, threshold=3):
    distance = calculate_levenshtein_distance(student_answer, correct_answer)
    if distance <= threshold:
        return f"Acceptable answer. Distance: {distance}"
    else:
        return f"Unacceptable answer. Distance: {distance}"

# Example usage
correct_answer = "The capital of France is Paris."
student_answer = "France is Pars capital."

evaluation = evaluate_answer(student_answer, correct_answer)
print(evaluation)


Unacceptable answer. Distance: 23


#### POS


In [163]:
def annotate_text(text):
    doc = nlp(text)
    annotated_text = []
    for token in doc:
        print(f"{token.text} {token.lemma_} {token.pos_} {token.tag_} {token.dep_} {token.shape_} {token.is_alpha} {token.is_stop}")
       
        if token.ent_type_:
            annotated_text.append(f"{token.text} <{token.ent_type_}>")  
        else:
            annotated_text.append(token.text)
    # print(annotated_text)
    return " ".join(annotated_text)


In [164]:
t="Apple is looking at buying U.K. startup for $1 billion"
s="Martha Bradley (fl. 1740s–1755) was a British cookery book writer. Little is known about her life, except that she published the cookery book The British Housewife (pictured) in 1756 and worked as a cook for more than 30 years in the fashionable spa town of Bath, Somerset. The British Housewife was released as a 42-issue partwork between January and October 1756. "
print(annotate_text(t))
print(annotate_text(s))


Apple Apple PROPN NNP nsubj Xxxxx True False
is be AUX VBZ aux xx True True
looking look VERB VBG ROOT xxxx True False
at at ADP IN prep xx True True
buying buy VERB VBG pcomp xxxx True False
U.K. U.K. PROPN NNP dobj X.X. False False
startup startup NOUN NN dep xxxx True False
for for ADP IN prep xxx True True
$ $ SYM $ quantmod $ False False
1 1 NUM CD compound d False False
billion billion NUM CD pobj xxxx True False
Apple <ORG> is looking at buying U.K. <GPE> startup for $ <MONEY> 1 <MONEY> billion <MONEY>
Martha Martha PROPN NNP compound Xxxxx True False
Bradley Bradley PROPN NNP nsubj Xxxxx True False
( ( PUNCT -LRB- punct ( False False
fl fl NOUN NN intj xx True False
. . PROPN NNP compound . False False
1740s–1755 1740s–1755 PROPN NNP pobj ddddx–dddd False False
) ) PUNCT -RRB- punct ) False False
was be AUX VBD ROOT xxx True True
a a DET DT det x True True
British british ADJ JJ amod Xxxxx True False
cookery cookery NOUN NN compound xxxx True False
book book NOUN NN compound xx

In [165]:

student_answer = "A collection of cheat sheets that will help you prepare for a technical interview, assessment tests, class presentation, and help you revise core data science concepts."
reference_answer = " collection of cheat sheets that will help you prepare for a technical interview,."

# Add annotations
student_answer_annotated = annotate_text(student_answer)
reference_answer_annotated = annotate_text(reference_answer)


sim(student_answer_annotated,reference_answer_annotated)



A a DET DT det X True True
collection collection NOUN NN ROOT xxxx True False
of of ADP IN prep xx True True
cheat cheat NOUN NN compound xxxx True False
sheets sheet NOUN NNS pobj xxxx True False
that that PRON WDT nsubj xxxx True True
will will AUX MD aux xxxx True True
help help VERB VB relcl xxxx True False
you you PRON PRP nsubj xxx True True
prepare prepare VERB VB ccomp xxxx True False
for for ADP IN prep xxx True True
a a DET DT det x True True
technical technical ADJ JJ amod xxxx True False
interview interview NOUN NN pobj xxxx True False
, , PUNCT , punct , False False
assessment assessment NOUN NN compound xxxx True False
tests test NOUN NNS conj xxxx True False
, , PUNCT , punct , False False
class class NOUN NN compound xxxx True False
presentation presentation NOUN NN conj xxxx True False
, , PUNCT , punct , False False
and and CCONJ CC cc xxx True True
help help VERB VB conj xxxx True False
you you PRON PRP nsubj xxx True True
revise revise VERB VB ccomp xxxx True False


,Model,Cosine Similarity Score,Answer Key,Student Answer,Embeddings Shape
2,BERT,0.907359,collection of cheat sheets that will help yo...,A collection of cheat sheets that will help yo...,"(768,)"
0,Sentence Transformer,0.835152,collection of cheat sheets that will help yo...,A collection of cheat sheets that will help yo...,"(384,)"
1,DistilBERT,0.811535,collection of cheat sheets that will help yo...,A collection of cheat sheets that will help yo...,"(768,)"


---

In [120]:
student_answer_annotated

'A collection of cheat sheets that will help you prepare for a technical interview , assessment tests , class presentation , and help you revise core data science concepts .'

In [76]:
for ent in doc.ents:
    print(ent.text, ent.label_)


In [82]:
nlp = spacy.load("en_core_web_sm")
a1="I am human"
a2="I am not human"
doc = nlp(a1)
doc2= nlp(a2)
doc.similarity(doc2)
print(doc.similarity(doc2))
sim(a1,a2)

0.7368373767096829
SENTENCE TRANSFORMER
--------------------

Answerkey: I am not human
Student Answer: I am human
Embeddings shape =(384,)
Cosine Similarity Score: 0.7448914051055908

DISTILBERT
----------

Answerkey: I am not human
Student Answer: I am human
Embeddings shape =(384,)
Cosine Similarity Score: 0.9190176129341125



/tmp/ipykernel_890/32248583.py:6: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  doc.similarity(doc2)
/tmp/ipykernel_890/32248583.py:7: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if av

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 9.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
# from sentence_transformers import SentenceTransformer
# import spacy
from nltk.metrics import edit_distance
from nltk.util import ngrams
from nltk.tokenize import word_tokenize

# Load models
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer('all-MiniLM-L6-v2')

def jaccard_similarity(text1, text2, n=1):
    set1 = set(ngrams(word_tokenize(text1.lower()), n))
    set2 = set(ngrams(word_tokenize(text2.lower()), n))
    return len(set1 & set2) / len(set1 | set2)

def extract_entities(text):
    doc = nlp(text)
    return {ent.text for ent in doc.ents}

student_answer = "Mitochondria generate ATP in eukaryotic cells."
reference_answer = "The mitochondria is responsible for ATP production in human cells."

# Compute embeddings
student_embedding = model.encode(student_answer)
reference_embedding = model.encode(reference_answer)

# Compute Similarity Metrics
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity([student_embedding], [reference_embedding])[0][0]

edit_dist = edit_distance(student_answer, reference_answer)
jaccard_score = jaccard_similarity(student_answer, reference_answer, n=2)

student_entities = extract_entities(student_answer)
reference_entities = extract_entities(reference_answer)
entity_match = len(student_entities & reference_entities) / len(reference_entities) if reference_entities else 0

# Final Score (Weighted Combination)
final_score = (0.5 * cosine_sim) + (0.2 * (1 - edit_dist / max(len(student_answer), len(reference_answer)))) + \
              (0.2 * jaccard_score) + (0.1 * entity_match)

print(f"Final Score: {final_score}")


ModuleNotFoundError: No module named 'nltk'

---

In [40]:
question_paper= {
    "part_a": [
      {"question_number": 1, "question": "What is machine learning?", "marks": 2},
      {"question_number": 2, "question": "Explain overfitting in neural networks.", "marks": 2},
      {"question_number": 3, "question": "Define supervised learning.", "marks": 2},
      {"question_number": 4, "question": "What is gradient descent?", "marks": 2},
      {"question_number": 5, "question": "Explain the difference between AI and ML.", "marks": 2}
    ],
    "part_b": [
      {"question_number": 1, "question": "Explain different types of activation functions.", "marks": 5},
      {"question_number": 2, "question": "What are the applications of convolutional neural networks?", "marks": 5}
    ],
    "part_c": [
      {"question_number": 1, "question": "Explain the working of backpropagation in neural networks with an example.", "marks": 10}
    ]
  }


In [54]:
answer_key= [(
"Machine learning is a subset of artificial intelligence that involves the use of algorithms and statistical models to enable computers to perform tasks without explicit instructions. It focuses on the development of programs that can access data and use it to learn for themselves."
"Overfitting in neural networks occurs when a model learns the training data too well, capturing noise and fluctuations rather than the underlying patterns. This leads to poor performance on new, unseen data. It can be mitigated through techniques such as regularization, dropout, and using more training data."
"Supervised learning is a type of machine learning where a model is trained on labeled data, meaning the input data is paired with the correct output. The model learns to map inputs to outputs based on this training data, allowing it to make predictions on new, unseen data."
"Gradient descent is an optimization algorithm used to minimize the loss function in machine learning models, particularly neural networks. It involves calculating the gradient (the slope) of the loss function with respect to the model parameters and adjusting the parameters in the opposite direction of the gradient to reduce the loss."
"Artificial Intelligence (AI) is a broader concept that encompasses the simulation of human intelligence in machines, enabling them to perform tasks that typically require human intelligence. Machine Learning (ML), on the other hand, is a subset of AI that focuses specifically on the development of algorithms that allow computers to learn from and make predictions based on data."
      ),
    
    ("Activation functions are mathematical functions that determine the output of a neural network node given an input or set of inputs. There are several types of activation functions, including:\n\n1. **Sigmoid Function**: Produces an output between 0 and 1, making it useful for binary classification tasks. However, it can cause vanishing gradient problems.\n\n2. **Tanh Function**: Similar to the sigmoid but outputs values between -1 and 1, helping to center the data. It also suffers from the vanishing gradient issue.\n\n3. **ReLU (Rectified Linear Unit)**: Outputs the input directly if it is positive; otherwise, it outputs zero. This helps to mitigate the vanishing gradient problem and is widely used in hidden layers of deep networks.\n\n4. **Softmax Function**: Converts a vector of values into probabilities, useful for multi-class classification tasks by ensuring that the total sum of the outputs equals 1.\n\nEach activation function has its strengths and weaknesses, and the choice of which to use can significantly impact the model's performance."
"Convolutional Neural Networks (CNNs) are particularly effective for tasks involving image processing and computer vision. Their applications include:\n\n1. **Image Classification**: Assigning labels to images based on their content, commonly used in applications like facial recognition and object detection.\n\n2. **Image Segmentation**: Dividing an image into segments for easier analysis, such as identifying different objects within an image or separating foreground from background.\n\n3. **Object Detection**: Identifying and localizing objects within images, useful in autonomous driving and surveillance systems.\n\n4. **Medical Image Analysis**: Assisting in diagnosing diseases from medical images like X-rays or MRIs by detecting abnormalities.\n\n5. **Video Analysis**: Analyzing video data for applications such as activity recognition and motion detection.\n\n6. **Image Generation**: Techniques like Generative Adversarial Networks (GANs) use CNNs for generating realistic images from noise.\n\nCNNs leverage the spatial structure of images, allowing them to achieve high levels of accuracy in these applications."
    ),
    
    ("Backpropagation is a supervised learning algorithm used for training neural networks. It involves the following steps:\n\n1. **Forward Pass**: The input data is passed through the network layer by layer, producing an output. The output is compared to the actual target output to calculate the loss using a loss function.\n\n2. **Calculating Gradients**: The loss is then propagated back through the network to compute the gradients of the loss with respect to each weight in the network using the chain rule of calculus. This process determines how much each weight contributed to the error.\n\n3. **Weight Update**: Using the computed gradients, the weights of the network are updated to minimize the loss. This is typically done using gradient descent or its variants, adjusting the weights in the opposite direction of the gradient.\n\n**Example**: Consider a simple neural network with one input layer, one hidden layer, and one output layer. When a training example is fed into the network:\n- The input values are transformed through the weights and activation functions, producing an output.\n- If the output differs from the target, the loss is calculated.\n- During backpropagation, the gradients of the loss are computed with respect to the weights in the hidden and input layers. This allows for fine-tuning of the weights in the next iteration of training, enabling the network to improve its predictions over time."
    )
  ]



In [42]:
len("Activation functions are mathematical functions that determine the output of a neural network node given an input or set of inputs. There are several types of activation functions, including:\n\n1. **Sigmoid Function**: Produces an output between 0 and 1, making it useful for binary classification tasks. However, it can cause vanishing gradient problems.\n\n2. **Tanh Function**: Similar to the sigmoid but outputs values between -1 and 1, helping to center the data. It also suffers from the vanishing gradient issue.\n\n3. **ReLU (Rectified Linear Unit)**: Outputs the input directly if it is positive; otherwise, it outputs zero. This helps to mitigate the vanishing gradient problem and is widely used in hidden layers of deep networks.\n\n4. **Softmax Function**: Converts a vector of values into probabilities, useful for multi-class classification tasks by ensuring that the total sum of the outputs equals 1.\n\nEach activation function has its strengths and weaknesses, and the choice of which to use can significantly impact the model's performance.")     

1051

In [43]:
len("Backpropagation is a supervised learning algorithm used for training neural networks. It involves the following steps:\n\n1. **Forward Pass**: The input data is passed through the network layer by layer, producing an output. The output is compared to the actual target output to calculate the loss using a loss function.\n\n2. **Calculating Gradients**: The loss is then propagated back through the network to compute the gradients of the loss with respect to each weight in the network using the chain rule of calculus. This process determines how much each weight contributed to the error.\n\n3. **Weight Update**: Using the computed gradients, the weights of the network are updated to minimize the loss. This is typically done using gradient descent or its variants, adjusting the weights in the opposite direction of the gradient.\n\n**Example**: Consider a simple neural network with one input layer, one hidden layer, and one output layer. When a training example is fed into the network:\n- The input values are transformed through the weights and activation functions, producing an output.\n- If the output differs from the target, the loss is calculated.\n- During backpropagation, the gradients of the loss are computed with respect to the weights in the hidden and input layers. This allows for fine-tuning of the weights in the next iteration of training, enabling the network to improve its predictions over time."
      )

1416

In [3]:
students = [
    {
      "student_id": "CSE2024001",
              "answers": [
                  "Machine learning is a part of AI that learns from data.",
                  "Overfitting means the model performs well on training data but poorly on unseen data.",
                  "Supervised learning is when the machine learns from labeled examples.",
                  "",
                  "AI is broader than ML, where ML is a subset focused on learning from data."
                ],
                 {
                  "1": "Activation functions include ReLU, Tanh, and Sigmoid. They help neurons decide.",
                  "2": "CNNs are applied in images and video processing."
                },
                "part_c": {
                  "1": "Backpropagation is used in neural networks to reduce error by adjusting weights backward based on gradients."
                }
      }
    },

    {
      "student_id": "AIDS2024005",
      "name": "Jane Smith",
      "department": "AIDS",
              "answers": {
                "part_a": {
                  "1": "Machine learning allows systems to learn from experience.",
                  "2": "Overfitting is when the model performs well on training data but not on test data.",
                  "3": "Supervised learning involves labeled data.",
                  "4": "Gradient descent is used to optimize the model.",
                  "5": "AI is general, ML is specific to learning from data."
                },
                "part_b": {
                  "1": "ReLU, Sigmoid, and Tanh are activation functions in neural networks.",
                  "2": "CNNs are used in image processing and facial recognition."
                },
                "part_c": {
                  "1": "Backpropagation updates the weights of a neural network to reduce error."
        }
      }
    },
    {
      "student_id": "ECE2024010",
      "name": "Alice Brown",
      "department": "ECE",
      "answers": {
        "part_a": {
          "1": "",
          "2": "Overfitting happens when the model fits the training data too well.",
          "3": "",
          "4": "Gradient descent helps in finding the minimum cost.",
          "5": ""
        },
        "part_b": {
          "1": "",
          "2": ""
        },
        "part_c": {
          "1": "Backpropagation updates the network weights based on error feedback."
        }
      }
    }
  ]



In [53]:
students= [{
                "student_id": "S001",
                    "answers": [
                            ("Machine learning is a technology that allows computers to learn from data and make decisions based on that data.",
                             "Overfitting happens when a model learns the details of the training data to an extent that it negatively impacts the performance on new data.",
                             "Supervised learning is when we train a model on labeled data, which means that we provide the correct output for every input.",
                             "Gradient descent is an optimization algorithm used to minimize the loss function in a model by iteratively adjusting the parameters.",
                             "AI is the broader concept of machines being able to carry out tasks in a way that we would consider 'smart', while ML is a specific subset of AI that focuses on the idea that we should really just be able to give machines access to data and let them learn for themselves.",
                            ),

                            ("Activation functions are vital components of neural networks that help to introduce non-linearity into the model, allowing it to learn complex patterns in the data. There are various types of activation functions, including:\n\n1. **Sigmoid Function**: This function outputs a value between 0 and 1, making it useful for binary classification. However, it can cause issues during training due to the vanishing gradient problem, where gradients become too small for effective learning.\n\n2. **Tanh Function**: This is similar to the sigmoid but outputs values between -1 and 1, which helps to center the data, leading to better convergence.\n\n3. **ReLU (Rectified Linear Unit)**: This function outputs the input directly if it is positive; otherwise, it outputs zero. It is the most commonly used activation function in deep learning because it reduces the likelihood of vanishing gradients and speeds up training significantly.\n\n4. **Softmax Function**: This function is often used in the output layer of multi-class classification problems. It converts logits into probabilities by ensuring the sum of all probabilities equals 1. \n\nChoosing the right activation function is crucial as it can affect the performance and convergence of the neural network.",
                             "Convolutional Neural Networks (CNNs) have revolutionized how we approach image processing and computer vision. They are designed to automatically and adaptively learn spatial hierarchies of features from input images. Here are some key applications:\n\n1. **Image Classification**: CNNs are extensively used for classifying images into categories. For instance, they can distinguish between cats and dogs in pictures.\n\n2. **Object Detection**: They can identify and locate objects within an image, such as finding faces in a crowd or detecting cars in traffic.\n\n3. **Image Segmentation**: CNNs are applied in segmenting images into meaningful parts, allowing for more granular analysis. For example, segmenting medical images to isolate tumors.\n\n4. **Facial Recognition**: Used in security systems and social media, CNNs help to identify individuals in images based on their facial features.\n\n5. **Video Analysis**: CNNs can be used in video data for activity recognition, such as identifying specific actions in a surveillance video.\n\n6. **Medical Image Analysis**: CNNs assist in diagnosing diseases by analyzing medical images like X-rays or MRIs. They can highlight anomalies, aiding doctors in making informed decisions.\n\nIn conclusion, the application of CNNs spans various fields, showcasing their versatility and efficiency in processing visual data."
                             ),

                            ("Backpropagation is a fundamental algorithm used to train neural networks. It works in two main phases: forward propagation and backward propagation.\n\n1. **Forward Propagation**: Initially, the input data is fed into the network, and it moves from the input layer through the hidden layers to the output layer. Each neuron applies a weighted sum of its inputs and then passes this through an activation function to produce an output. The outputs are compared against the true labels using a loss function to compute the error.\n\n2. **Backward Propagation**: The next step involves calculating the gradient of the loss function with respect to each weight in the network by applying the chain rule. Starting from the output layer, we compute how much each weight contributed to the error. These gradients are then used to update the weights in the direction that minimizes the error. This is often done using an optimization algorithm like gradient descent.\n\n**Example**: Consider a simple neural network with one hidden layer. Let's say we have an input (e.g., an image) and we want to predict its class (e.g., cat or dog). During the forward pass, the input data is processed, and the network generates an output. After comparing this output with the true label, we compute the loss. In the backward pass, we calculate the gradients for each weight based on the loss and adjust them accordingly to improve the model's accuracy in subsequent iterations. Through repeated cycles of forward and backward propagation, the model learns to make better predictions, adapting its weights to reduce errors over time."
                             )
                            ]
            },
            {
                    "student_id": "S002",
                    "answers": [
                                ("Machine learning is when machines learn from data.",
                                "Overfitting is when a model is too complex and learns the noise.",
                                "Supervised learning uses labeled data for training.",
                                "Gradient descent is a way to minimize the loss function.",
                                "AI is the big idea, while ML is a part of AI."
                                ),
                                
                                ("There are many types of activation functions in neural networks that help to decide how to process information. One is the sigmoid function, which outputs numbers between 0 and 1. It is good for binary classification but can slow down learning because of the vanishing gradient problem. Another function is tanh, which outputs between -1 and 1. This is better than sigmoid because it centers data around zero.\n\nThe ReLU function, which stands for Rectified Linear Unit, is widely used. It outputs the input directly if it's positive; otherwise, it outputs zero. This helps in speeding up learning. Lastly, the softmax function is used in multi-class classification problems. It takes a vector of scores and converts them into probabilities, making it easier to classify multiple classes. In summary, the choice of activation function can greatly affect how well a neural network learns.",
                                 "CNNs are used for many things in image processing. They are very useful in image classification where the goal is to identify what an image contains. They can also be used for object detection, meaning they can find where objects are located within an image. Another use is in image segmentation, which is breaking an image into different parts to analyze.\n\nFacial recognition is a popular application, enabling security systems to identify people based on their facial features. Moreover, CNNs are useful in analyzing medical images, helping doctors diagnose diseases by highlighting anomalies. In video analysis, CNNs can identify actions or behaviors over time. Overall, CNNs are vital tools for analyzing visual data."
                                ),

                                ("Backpropagation is an algorithm used to train neural networks. It works by first calculating the output of the network, comparing it to the actual target output, and computing the error. Then, it uses this error to update the weights in the network.\n\nIn simple terms, the process begins with a forward pass where the input data moves through the network to generate an output. Then, the loss is calculated. After that, in the backward pass, the algorithm computes how much each weight contributed to the loss by calculating gradients.\n\nFor instance, in a neural network with input, hidden, and output layers, when an image is processed, the output will be compared to the actual label. The difference is the error. This error is then backpropagated through the layers to adjust the weights to improve accuracy. Over time, through many iterations, the network learns to make better predictions."
                                )   
                        ]
                    
            },
                
            {    "student_id": "S003",
                    "answers": [(
                                "Machine learning is a process by which computers analyze data and learn from it.",
                                "Overfitting is a significant issue in machine learning where a model performs well on training data but poorly on unseen data.",
                                "Supervised learning involves training a model on a labeled dataset.",
                                "Gradient descent is an iterative optimization algorithm used for minimizing a function.",
                                "AI encompasses a wide range of technologies, while ML specifically focuses on algorithms that improve with experience."
                                 ),

                                (
                                "Activation functions are crucial for neural networks as they introduce non-linearities into the model, enabling it to learn complex relationships in the data. Common activation functions include:\n\n1. **Sigmoid**: It squashes the output to be between 0 and 1, making it ideal for binary classification tasks. However, its major drawback is the vanishing gradient problem, where gradients become very small for large inputs, slowing down learning.\n\n2. **Tanh**: This function is similar to the sigmoid but outputs values between -1 and 1. It is generally preferred over sigmoid because it has a mean of zero, which helps in faster convergence during training.\n\n3. **ReLU (Rectified Linear Unit)**: ReLU is the most commonly used activation function. It outputs the input directly if it is positive; otherwise, it outputs zero. This allows for faster training and helps to mitigate the vanishing gradient issue, making it a popular choice in deep networks.\n\n4. **Softmax**: This function converts raw scores (logits) from the last layer of the network into probabilities, ensuring that the total probability across all classes equals 1. It is particularly useful in multi-class classification problems.\n\nIn summary, the choice of activation function impacts the learning process and the performance of the neural network.",
                                "Convolutional Neural Networks (CNNs) have a wide range of applications, especially in the field of computer vision. They are designed to process data with a grid-like topology, such as images. Here are some prominent applications:\n\n1. **Image Classification**: CNNs can categorize images into different classes, such as distinguishing between dogs and cats based on learned features.\n\n2. **Object Detection**: They can identify and locate objects within images, a task crucial for autonomous vehicles and surveillance systems.\n\n3. **Image Segmentation**: This process involves dividing an image into segments for more granular analysis, such as identifying different regions in medical images.\n\n4. **Facial Recognition**: CNNs are widely used in security systems for recognizing individuals based on facial features, which has applications in both personal and public safety.\n\n5. **Medical Imaging**: They play a vital role in analyzing X-rays, MRIs, and CT scans to assist in diagnosing diseases, highlighting any abnormalities.\n\n6. **Video Analysis**: CNNs are used in analyzing video data for various applications, including action recognition in sports or identifying suspicious activities in surveillance footage.\n\nIn conclusion, CNNs have transformed how we process and analyze visual data, making them invaluable in many fields."
                                ),    
                                
                                ("Backpropagation is a key algorithm in training neural networks. It consists of two main phases: forward propagation and backward propagation. The forward pass involves sending the input data through the network to generate an output. Once the output is obtained, it is compared to the actual target output to compute the loss using a predefined loss function.\n\nIn the backward pass, the algorithm computes the gradient of the loss function with respect to each weight in the network. This is done using the chain rule of calculus, allowing the model to determine how much each weight contributed to the loss. The calculated gradients are then used to update the weights in the network through an optimization algorithm, typically gradient descent, where weights are adjusted in the direction that reduces the loss.\n\n**Example**: Consider a neural network designed to recognize handwritten digits. When an image of a handwritten digit is fed into the network, it goes through the forward pass, and the network outputs a prediction. If the prediction is incorrect, the loss is calculated, and during the backpropagation step, the algorithm computes the gradients based on this loss. The weights are then updated to improve the accuracy of the network's predictions over successive training iterations. Through many epochs of this process, the network learns to recognize handwritten digits accurately."
                                )
                                ]
            },
            {                      
                "student_id": "S004",
                    "answers":
                                [( "Machine learning is when computers are trained to learn from data.",
                                    "Overfitting is a problem where the model learns the training data too well.",
                                    "Supervised learning is when a model is trained on labeled data.",
                                    "Gradient descent is a method to reduce error in models.",
                                    "AI is the concept while ML is a method of achieving AI."
                                ),
                                
                                (
                                    "Activation functions are an essential component in neural networks that determine how signals are transformed and passed through the layers. Different types of activation functions are designed to handle various tasks, enabling the network to learn complex patterns in data.\n\n1. **Sigmoid**: This function outputs a value between 0 and 1, making it suitable for binary classification tasks. However, it suffers from the vanishing gradient problem, where gradients can become very small, slowing down learning.\n\n2. **Tanh**: Similar to the sigmoid function, tanh outputs values between -1 and 1, helping to center the data and often resulting in faster convergence compared to sigmoid.\n\n3. **ReLU (Rectified Linear Unit)**: ReLU has gained popularity in deep learning because it outputs the input directly if it is positive and zero otherwise. This alleviates the vanishing gradient issue and accelerates convergence, allowing networks to learn efficiently.\n\n4. **Softmax**: Used in multi-class classification tasks, the softmax function converts logits into probabilities, ensuring that the total probability across all classes equals 1.\n\nIn summary, the choice of activation function can significantly impact the performance and convergence speed of a neural network.",
                                    "CNNs are powerful tools in the field of image processing. They can be applied in various domains, and their effectiveness lies in their ability to automatically learn features from data. Some of the main applications include:\n\n1. **Image Classification**: CNNs are used to classify images based on their content. For instance, they can distinguish between various objects, such as identifying animals or objects in photographs.\n\n2. **Object Detection**: These networks can locate and identify objects within an image. This capability is essential in applications such as autonomous driving, where recognizing pedestrians and other vehicles is critical.\n\n3. **Image Segmentation**: CNNs can divide images into segments to analyze different regions independently. This is particularly useful in medical imaging, where identifying specific areas can aid in diagnosis.\n\n4. **Facial Recognition**: CNNs are employed in security systems for identifying individuals based on their facial features, which is widely used in surveillance and personal devices.\n\n5. **Medical Imaging**: CNNs help analyze medical images, assisting in diagnosing diseases by highlighting anomalies or areas of concern.\n\nIn conclusion, the applications of CNNs are extensive, and they have significantly advanced the field of computer vision."
                                ),
                                ("Backpropagation is a critical algorithm in training neural networks, consisting of two primary stages: forward and backward propagation. In the forward pass, input data is fed through the network, generating an output that is compared against the true target to compute the loss.\n\nIn the backward pass, the algorithm calculates the gradient of the loss function with respect to each weight in the network using the chain rule of calculus. This step determines how much each weight contributed to the error. The calculated gradients are then used to update the weights in the network, typically employing an optimization technique like gradient descent to minimize the loss.\n\n**Example**: Consider a neural network designed to predict housing prices based on various input features. During training, input data representing different houses is processed through the network to produce predicted prices. After calculating the loss by comparing predicted prices to actual prices, backpropagation is used to compute gradients and update the model weights. Through repeated iterations of this process, the network learns to make accurate predictions about housing prices."
                                )]
            }
            ]


In [53]:
answer_info['question_number']

[{'question_number': 1,
  'answer': 'Machine learning is when computers are trained to learn from data.'},
 {'question_number': 2,
  'answer': 'Overfitting is a problem where the model learns the training data too well.'},
 {'question_number': 3,
  'answer': 'Supervised learning is when a model is trained on labeled data.'},
 {'question_number': 4,
  'answer': 'Gradient descent is a method to reduce error in models.'},
 {'question_number': 5,
  'answer': 'AI is the concept while ML is a method of achieving AI.'}]

In [54]:
def encode_answer(answer):
    """Encodes a student's answer using the model."""
    return model.encode(answer)

def calculate_similarity(student_answer, answer_key_answer):
    """Calculates cosine similarity between student's answer and answer key."""
    student_embedding = encode_answer(student_answer)
    answer_key_embedding = encode_answer(answer_key_answer)
    similarity = cosine_similarity([student_embedding], [answer_key_embedding])[0][0]
    return similarity

def process_answers(part_answers, answer_key, part_weight):
    """Processes answers for a specific part and calculates marks."""
    total_marks = 0
    marks = {}
    
    for answer_info in part_answers:
        q_num = answer_info["question_number"] - 1  # Adjust for zero-based index
        answer = answer_info["answer"]
        
        if answer:  # Check if answer is not empty
            try:
                similarity = calculate_similarity(answer, answer_key[q_num])
                marks_for_question = round(similarity * part_weight, 2)  # Assign marks based on similarity
                marks[f"q{answer_info['question_number']}"] = marks_for_question
                total_marks += marks_for_question
            except KeyError:
                print(f"KeyError for question number: {answer_info['question_number']}")
                marks[f"q{answer_info['question_number']}"] = 0
        else:
            marks[f"q{answer_info['question_number']}"] = 0  # No marks for empty answers

    return total_marks, marks

def grade_student(student, answer_key):
    """Grades a single student's answers across all parts."""
    student_results = {
        "student_id": student["student_id"],
        "name": student["name"],
        "department": student["department"],
        "marks": {}
    }
    
    # Process Part A
    total_part_a, part_a_marks = process_answers(student["answers"]["part_a"], answer_key["part_a"], 2)
    student_results["marks"].update(part_a_marks)
    student_results["total_part_a"] = total_part_a
    
    # Process Part B
    total_part_b, part_b_marks = process_answers(student["answers"]["part_b"], answer_key["part_b"], 5)
    student_results["marks"].update(part_b_marks)
    student_results["total_part_b"] = total_part_b

    # Process Part C
    total_part_c, part_c_marks = process_answers(student["answers"]["part_c"], answer_key["part_c"], 10)
    student_results["marks"].update(part_c_marks)
    student_results["total_part_c"] = total_part_c
    
    # Calculate total marks
    student_results["total_marks"] = total_part_a + total_part_b + total_part_c
    return student_results

def grade_students(students, answer_key):
    """Grades all students and compiles results into a DataFrame."""
    results = []
    for student in students:
        student_results = grade_student(student, answer_key)
        results.append(student_results)

    # Create a DataFrame from results
    df_results = pd.DataFrame(results)

    # Sort students by total marks in descending order
    df_results = df_results.sort_values(by="total_marks", ascending=False).reset_index(drop=True)

    return df_results

# Example usage
df_results = grade_students(students, answer_key)

# Display the DataFrame
print(df_results)


KeyError for question number: 1
KeyError for question number: 2
KeyError for question number: 3
KeyError for question number: 4
KeyError for question number: 5
KeyError for question number: 1
KeyError for question number: 2
KeyError for question number: 1
KeyError for question number: 1
KeyError for question number: 2
KeyError for question number: 3
KeyError for question number: 4
KeyError for question number: 5
KeyError for question number: 1
KeyError for question number: 2
KeyError for question number: 1
KeyError for question number: 1
KeyError for question number: 2
KeyError for question number: 3
KeyError for question number: 4
KeyError for question number: 5
KeyError for question number: 1
KeyError for question number: 2
KeyError for question number: 1
KeyError for question number: 1
KeyError for question number: 2
KeyError for question number: 3
KeyError for question number: 4
KeyError for question number: 5
KeyError for question number: 1
KeyError for question number: 2
KeyError

In [59]:
def calculate_similarity(student_answer, answer_key_answer):
    """Calculates cosine similarity between student's answer and answer key."""
    student_embedding = encode_answer(student_answer)
    answer_key_embedding = encode_answer(answer_key_answer)
    
    # Debug: Print embeddings
    print("Student Embedding:", student_embedding)
    print("Answer Key Embedding:", answer_key_embedding)
    
    similarity = cosine_similarity([student_embedding], [answer_key_embedding])[0][0]
    
    # Debug: Print similarity
    print("Similarity:", similarity)
    
    return similarity

def process_answers(part_answers, answer_key, part_weight):
    """Processes answers for a specific part and calculates marks."""
    total_marks = 0
    marks = {}
    
    for answer_info in part_answers:
        q_num = answer_info["question_number"] - 1  # Adjust for zero-based index
        answer = answer_info["answer"]
        
        # Debug: Print the current question number and answer
        print(f"Processing Part Answer - Question: {answer_info['question_number']}, Answer: {answer}")
        
        if answer:  # Check if answer is not empty
            try:
                similarity = calculate_similarity(answer, answer_key[q_num])
                marks_for_question = round(similarity * part_weight, 2)  # Assign marks based on similarity
                marks[f"q{answer_info['question_number']}"] = marks_for_question
                total_marks += marks_for_question
                
                # Debug: Print marks for the question
                print(f"Marks for Question {answer_info['question_number']}: {marks_for_question}")
                
            except KeyError:
                print(f"KeyError for question number: {answer_info['question_number']}")
                marks[f"q{answer_info['question_number']}"] = 0
        else:
            marks[f"q{answer_info['question_number']}"] = 0  # No marks for empty answers

    return total_marks, marks


In [60]:
df_results = grade_students(students, answer_key)

Processing Part Answer - Question: 1, Answer: Machine learning is a technology that allows computers to learn from data and make decisions based on that data.
KeyError for question number: 1
Processing Part Answer - Question: 2, Answer: Overfitting happens when a model learns the details of the training data to an extent that it negatively impacts the performance on new data.
KeyError for question number: 2
Processing Part Answer - Question: 3, Answer: Supervised learning is when we train a model on labeled data, which means that we provide the correct output for every input.
KeyError for question number: 3
Processing Part Answer - Question: 4, Answer: Gradient descent is an optimization algorithm used to minimize the loss function in a model by iteratively adjusting the parameters.
KeyError for question number: 4
Processing Part Answer - Question: 5, Answer: AI is the broader concept of machines being able to carry out tasks in a way that we would consider 'smart', while ML is a speci

IndexError: list index out of range

In [ ]:
def grade_student(student, answer_key):
    """Grades a single student's answers across all parts."""
    student_results = {
        "student_id": student["student_id"],
        "name": student["name"],
        "department": student["department"],
        "marks": {}
    }
    
    # Process Part A
    total_part_a, part_a_marks = process_answers(student["answers"]["part_a"], answer_key["part_a"], 2)
    student_results["marks"].update(part_a_marks)
    student_results["total_part_a"] = total_part_a
    
    # Process Part B
    total_part_b, part_b_marks = process_answers(student["answers"]["part_b"], answer_key["part_b"], 5)
    student_results["marks"].update(part_b_marks)
    student_results["total_part_b"] = total_part_b

    # Process Part C
    total_part_c, part_c_marks = process_answers(student["answers"]["part_c"], answer_key["part_c"], 10)
    student_results["marks"].update(part_c_marks)
    student_results["total_part_c"] = total_part_c
    
    # Calculate total marks
    student_results["total_marks"] = total_part_a + total_part_b + total_part_c
    return student_results

def process_answers(part_answers, answer_key, part_weight):
    """Processes answers for a specific part and calculates marks."""
    total_marks = 0
    marks = {}
    
    for answer_info in part_answers:
        q_num = answer_info["question_number"] - 1  # Adjust for zero-based index
        answer = answer_info["answer"]
        
        # Debug: Print the current question number and answer
        print(f"Processing Part Answer - Question: {answer_info['question_number']}, Answer: {answer}")

grade_student(student, answer_key)


In [14]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import os
import json

# Initialize the sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')


/home/irshad/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [49]:
def save_output(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f)
    print(f"Saved output to {filename}")

def convert_to_embeddings(data):
    print("Starting embedding conversion...")
    embeddings = model.encode(data)
    print("Finished embedding conversion.")
    return embeddings

def calculate_cosine_similarity(answer_embedding, key_embedding):
    print("Calculating cosine similarity...")
    similarity = cosine_similarity([answer_embedding], [key_embedding])[0][0]
    print(f"Cosine similarity: {similarity}")
    return similarity

def score_from_similarity(similarity, max_marks):
    print("Converting cosine similarity to score...")
    score = similarity * max_marks  # Simple scaling
    print(f"Score: {score} out of {max_marks}")
    return score

def calculate_total_marks(student_scores):
    
    print("Calculating total marks for student...")
    total_marks = sum(student_scores)
    print(f"Total marks: {total_marks}")
    return total_marks

def process_student(student, answer_key, save_intermediate):
    scores = []

    print(f" Inside process_student \n\nstudent = {student} \n scores ={scores}")
    for part, questions in answer_key.items():
       
        print(f"\n\█▓▒▒░░░PART░░░▒▒▓█{part}\n\n \
                            questions: {questions} \n\n\
                                  from answer_key.items {answer_key.items()} \n\n")
        
        for question in questions:
            print(f"question: {question}\n\n")
            question_number = str(question["question_number"])
            print(f"str (question_number): {question_number}\n\n\n\n")
            print("getting ans from ",student["answers"],"\n\n Printing ",part,"\n\n",student["answers"][part] ,"\n")
            answer = student["answers"][part][0].get(question_number, "")
            print(f"str (answer): {answer}")

            if answer:  # Only process if answer is provided
                embedding_answer = convert_to_embeddings([answer])[0]
                print("embedding_answer = \n ",embedding_answer)
                embedding_key = convert_to_embeddings([question["answer"]])[0]
                print("embedding_key = \n ",embedding_key)
                similarity = calculate_cosine_similarity(embedding_answer, embedding_key)
                print("similarity=", similarity)
                score = score_from_similarity(similarity, question["marks"])
                print(score)
                scores.append(score)
                print(scores)
            else:
                scores.append(0)  # No answer results in 0 score

            if save_intermediate:
                save_output(scores, f'intermediate_scores_{student["student_id"]}.json')

    total_marks = calculate_total_marks(scores)
    return scores, total_marks

def calculate_results(students, answer_key, save_intermediate=False):
    results = []
    for student in students:
        print(f"✌𝓟𝓻𝓸𝓬𝓮𝓼𝓼𝓲𝓷𝓰 𝓼𝓽𝓾𝓭𝓮𝓷𝓽✌: {student['student_id']}")
        scores, total_marks = process_student(student, answer_key, save_intermediate)
        result = {
            "student_id": student["student_id"],
            "scores": scores,
            "total_marks": total_marks
        }
        results.append(result)
    
    final_df = pd.DataFrame(results)
    print("Final results DataFrame created.")
    if not save_intermediate:
        final_df.to_csv('final_results.csv', index=False)
        print("Saved final results to final_results.csv")
    return final_df

# Example usage:
# final_results = calculate_results(students, answer_key, save_intermediate=True)


In [50]:
final_results = calculate_results(students, answer_key, save_intermediate=True)


✌𝓟𝓻𝓸𝓬𝓮𝓼𝓼𝓲𝓷𝓰 𝓼𝓽𝓾𝓭𝓮𝓷𝓽✌: S001
 Inside process_student 

student = {'student_id': 'S001', 'name': 'Alice Johnson', 'department': 'Computer Science', 'answers': {'part_a': [{'question_number': 1, 'answer': 'Machine learning is a technology that allows computers to learn from data and make decisions based on that data.'}, {'question_number': 2, 'answer': 'Overfitting happens when a model learns the details of the training data to an extent that it negatively impacts the performance on new data.'}, {'question_number': 3, 'answer': 'Supervised learning is when we train a model on labeled data, which means that we provide the correct output for every input.'}, {'question_number': 4, 'answer': 'Gradient descent is an optimization algorithm used to minimize the loss function in a model by iteratively adjusting the parameters.'}, {'question_number': 5, 'answer': "AI is the broader concept of machines being able to carry out tasks in a way that we would consider 'smart', while ML is a specific subs

In [10]:
students[0].keys()

dict_keys(['student_id', 'name', 'department', 'answers'])